In [9]:
# !conda install -c conda-forge poppler
!brew install poppler

==> Downloading https://formulae.brew.sh/api/formula.jws.json
######################################################################### 100.0%
==> Downloading https://formulae.brew.sh/api/cask.jws.json
######################################################################### 100.0%
==> Downloading https://ghcr.io/v2/homebrew/core/poppler/manifests/24.11.0
######################################################################### 100.0%
==> Fetching dependencies for poppler: libgpg-error, libassuan, libgcrypt, libksba, libusb, npth, pinentry, gnupg, gpgme, nspr and nss
==> Downloading https://ghcr.io/v2/homebrew/core/libgpg-error/manifests/1.51
######################################################################### 100.0%
==> Fetching libgpg-error
==> Downloading https://ghcr.io/v2/homebrew/core/libgpg-error/blobs/sha256:cb513
######################################################################### 100.0%
==> Downloading https://ghcr.io/v2/homebrew/core/libassuan/manifests/3.0.1
#####

In [7]:
from pdf2image import convert_from_path
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image

def extract_text_from_pdf_with_huggingface(pdf_path):
    """
    Extracts text from a PDF document using Hugging Face's TrOCR model.

    Args:
        pdf_path (str): Path to the PDF file.

    Returns:
        str: Extracted text from the PDF.
    """
    try:
        # Initialize Hugging Face model and processor
        processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-stage1")
        model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-stage1")

        # Convert PDF to images
        images = convert_from_path(pdf_path)
        extracted_text = ""

        for image in images:
            # Resize image if needed (optional)
            image = image.convert("RGB")  # Ensure it's RGB format
            # Perform OCR using Hugging Face TrOCR
            pixel_values = processor(images=image, return_tensors="pt").pixel_values
            generated_ids = model.generate(pixel_values)
            text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

            # Append the extracted text
            extracted_text += text + "\n"

        return extracted_text

    except Exception as e:
        print(f"An error occurred: {e}")
        return ""


/Users/brinkley97/opt/anaconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Example usage
pdf_path = "datasets/nsbe_renamed/cornerstone-1974-1stEditionEditorGeorgeSmith.pdf"  # Replace with your PDF file path
pdf_text = extract_text_from_pdf_with_huggingface(pdf_path)
print(pdf_text)

2024-11-24 00:17:06.151590: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-stage1 and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


An error occurred: Unable to get page count. Is poppler installed and in PATH?

